In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from models.clp_transformer import CLPTransformer
from training.training import load_model

model_name = "ejemplo"

model = load_model(CLPTransformer, model_name)

# Configuramos el adaptador para la entrada del modelo
from data.adapters.input.v1 import InputAdapterV1

input_adapter = InputAdapterV1(max_blocks=10000, max_pblocks=64, max_actions=64)

# Solvers

## Greedy

In [13]:
from solvers.greedy import GreedyModelSolver, VCSSolver

instance_file = "benchmarks/BR4.txt"
instance_number = 0
min_fr = 1

greedy_solver = GreedyModelSolver(model, w=8, input_adapter=input_adapter, min_fr=min_fr) # W = 8 selecciona 64 candidatos
greedy_eval, _ = greedy_solver.solve(instance_file, instance_number)

vcs_solver = VCSSolver(min_fr=min_fr)
vcs_eval, _ = vcs_solver.solve(instance_file, instance_number)

print("GreedyModel:", f'{greedy_eval:.2f}')
print("VCSSolver:", f'{vcs_eval:.2f}')

GreedyModel: 87.60
VCSSolver: 92.03


## Beam Search

In [16]:
from solvers.bs import BSM_GM_Solver, BSM_VCS_Solver, BSG_Solver

instance_file = "benchmarks/BR8.txt"
instance_number = 0
w = 8
min_fr = 0.98

bsm_gm_solver = BSM_GM_Solver(model)
bsm_gm_eval, _ = bsm_gm_solver.solve(instance_file, instance_number, w, min_fr)

bsm_vcs_solver = BSM_VCS_Solver(model)
bsm_vcs_eval, _ = bsm_vcs_solver.solve(instance_file, instance_number, w, min_fr)

bsg_solver = BSG_Solver()
bsg_eval, _ = bsg_solver.solve(instance_file, instance_number, w, min_fr)

print("BSM GM:", f'{bsm_gm_eval:.2f}')
print("BSM VCS:", f'{bsm_vcs_eval:.2f}')
print("BSG:", f'{bsg_eval:.2f}')

BSM GM: 95.73
BSM VCS: 95.50
BSG: 95.05


## Double Effort

In [17]:
from solvers.dse import DSE_BSM_GM_Solver, DSE_BSM_VCS_Solver, DSE_BSG_Solver

instance_file = "benchmarks/BR8.txt"
instance_number = 0
min_fr = 0.98
max_w = 8

dse_bsm_gm_solver = DSE_BSM_GM_Solver(model)
dse_bsm_gm_eval, _ = dse_bsm_gm_solver.solve(instance_file, instance_number, min_fr, max_w)

dse_bsm_vcs_solver = DSE_BSM_VCS_Solver(model)
dse_bsm_vcs_eval, _ = dse_bsm_vcs_solver.solve(instance_file, instance_number, min_fr, max_w)

dse_bsg_solver = DSE_BSG_Solver()
dse_bsg_eval, _ = dse_bsg_solver.solve(instance_file, instance_number, min_fr, max_w)

print("DSE BSM GM:", f'{dse_bsm_gm_eval:.2f}')
print("DSE BSM VCS:", f'{dse_bsm_vcs_eval:.2f}')
print("DSE BSG:", f'{dse_bsg_eval:.2f}')

DSE BSM GM: 95.73
DSE BSM VCS: 95.50
DSE BSG: 95.05


## Timed

In [18]:
from solvers.timed import Timed_BSM_GM_Solver, Timed_BSM_VCS_Solver, Timed_BSG_Solver

instance_file = "benchmarks/BR8.txt"
instance_number = 0
min_fr = 0.98
time = 10

timed_bsm_gm_solver = Timed_BSM_GM_Solver(model)
timed_bsm_gm_eval = timed_bsm_gm_solver.solve(instance_file, instance_number, min_fr, time)

timed_bsm_vcs_solver = Timed_BSM_VCS_Solver(model)
timed_bsm_vcs_eval = timed_bsm_vcs_solver.solve(instance_file, instance_number, min_fr, time)

timed_bsg_solver = Timed_BSG_Solver()
timed_bsg_eval = timed_bsg_solver.solve(instance_file, instance_number, min_fr, time)

print("Timed BSM GM:", f'{timed_bsm_gm_eval:.2f}')
print("Timed BSM VCS:", f'{timed_bsm_vcs_eval:.2f}')
print("Timed BSG:", f'{timed_bsg_eval:.2f}')

Timed BSM GM: 95.04
Timed BSM VCS: 95.11
Timed BSG: 94.67


In [6]:
from solvers.timed.timed_bsm_gm import Timed_BSM_GM_Solver

instance_file = "benchmarks/BR4.txt"
instance_number = 1
min_fr = 1
time = 3

bsm_solver = Timed_BSM_GM_Solver(model)
bsm_solver.verbose = True
bsm_eval = bsm_solver.solve(instance_file, instance_number, min_fr, time)
print(bsm_eval)

Ejecutando Timed BSM-GM con w = 1
Actualizando mejor volumen: 88.95985725309924
Ejecutando Timed BSM-GM con w = 2
Actualizando mejor volumen: 93.61347534465374
Ejecutando Timed BSM-GM con w = 3
Actualizando mejor volumen: 95.16375746852236
Ejecutando Timed BSM-GM con w = 4
95.16375746852236


# Comparación

In [9]:
from solvers.evaluators.greedy_evaluator import greedy_eval
from solvers.greedy import GreedyModelSolver, VCSSolver

greedy_model_solver = GreedyModelSolver(model, w=8)
vcs_solver = VCSSolver()

solver_list = [greedy_model_solver, vcs_solver]
instance_file = "benchmarks/BR8.txt"
num_instances = 3
min_fr = 0.98

df = greedy_eval(solver_list, instance_file, num_instances, min_fr)

Solver          | Instancia  | Avg Vol      | Avg Time    
----------------------------------------------------------
GreedyModel     | 3          | 92.34        | 0.11        
VCS             | 3          | 91.64        | 0.02        


In [7]:
from solvers.evaluators import bs_eval
from solvers.bs import BSM_GM_Solver, BSM_VCS_Solver, BSG_Solver

bsm_gm_solver = BSM_GM_Solver(model)
bsm_vcs_solver = BSM_VCS_Solver(model)
bsg_solver = BSG_Solver()

solver_list = [bsm_gm_solver, bsm_vcs_solver, bsg_solver]
instance_file = "benchmarks/BR8.txt"
num_instances = 3
w = 8
min_fr = 0.98

df = bs_eval(solver_list, instance_file, num_instances, w, min_fr)

Solver          | Instancia  | Avg Vol      | Avg Time    
----------------------------------------------------------
BSM-GM          | 3          | 95.71        | 14.22       
BSM-VCS         | 3          | 95.41        | 5.78        
BSG             | 3          | 95.10        | 5.99        


In [8]:
from solvers.evaluators import dse_eval
from solvers.dse import DSE_BSM_GM_Solver, DSE_BSM_VCS_Solver, DSE_BSG_Solver

dse_bsm_gm_solver = DSE_BSM_GM_Solver(model)
dse_bsm_vcs_solver = DSE_BSM_VCS_Solver(model)
dse_bsg_solver = DSE_BSG_Solver()

solver_list = [dse_bsm_gm_solver, dse_bsm_vcs_solver, dse_bsg_solver]
instance_file = "benchmarks/BR8.txt"
num_instances = 3
max_w = 4
min_fr = 0.98

df = dse_eval(solver_list, instance_file, num_instances, max_w, min_fr)

Solver          | Instancia  | Avg Vol      | Avg Time    
----------------------------------------------------------
DSE BSM-GM      | 3          | 95.01        | 3.52        
DSE BSM-VCS     | 3          | 94.49        | 1.63        
DSE BSG         | 3          | 94.49        | 1.34        


In [ ]:
from solvers.evaluators import timed_eval
from solvers.timed import Timed_BSM_GM_Solver, Timed_BSM_VCS_Solver, Timed_BSG_Solver

tm_bsm_gm_solver = Timed_BSM_GM_Solver(model)
tm_bsm_vcs_solver = Timed_BSM_VCS_Solver(model)
tm_bsg_solver = Timed_BSG_Solver()

solver_list = [tm_bsm_gm_solver, tm_bsm_vcs_solver, tm_bsg_solver]
instance_file = "benchmarks/BR8.txt"
num_instances = 3
min_fr = 0.98
time = 5

df = timed_eval(solver_list, instance_file, num_instances, min_fr, time)

Solver          | Instancia  | Avg Vol     
-------------------------------------------
Timed BSM-GM    | 3          | 95.01       
Timed BSM-VCS   | 3          | 95.12       
Timed BSG       | 3          | 94.76       
